# YRBSS Data Cleaning

This notebook cleans and prepares the 2023 Youth Risk Behavior Survey (YRBS) dataset for analysis.

Goals:

- Inspect the raw YRBSS data structure
- Identify the variables relevant to pediatric mental health and behavioral risk
- Review missing values and survey-specific coded responses
- Preserve the original raw file while creating an analysis-ready dataset
- Export the cleaned YRBSS dataset for later analysis and visualization

In [1]:
from pathlib import Path

yrbss_path = Path("../data/raw/YRBSS/XXH2023_YRBS_Data.dat")

with open(yrbss_path, "r", encoding="utf-8", errors="replace") as f:
    for _ in range(5):
        print(repr(f.readline()))

'XX              311   C     1.65 81.65441111311122122122122112111211111111111112111 112111111121212311112412216221131 2 112113313121233222                                              11  2212222222222212222222 22 22 2222 2221222 222222      212211112221121221222 2 2212 2221221222221                                                                 222222222222    2121222222122222222 12    0.8614103 1629497.08    505180\n'
'XX              4212    E             5111111122221223122221121112111111111111111111112111111111411111112556214222351422221113782522225212                                              22  2222112222211222222222 22 22 2222 22222222222222      1111111122211222222122222222 2211122211211                                                                 222222222222    12212222222212221112      0.8920103 16294  .     5N N233\n'
'XX              5232    E   1.68 74.845323111111221221221221115111221136111111111211111743323431413212124884118221211323221121611522124221         

In [2]:
# Check whether records use a consistent fixed width

with open(yrbss_path, "r", encoding="utf-8", errors="replace") as f:
    sample_lines = [f.readline().rstrip("\n") for _ in range(10)]

for i, line in enumerate(sample_lines, start=1):
    print(f"Row {i}: {len(line)} characters")

Row 1: 421 characters
Row 2: 421 characters
Row 3: 421 characters
Row 4: 421 characters
Row 5: 421 characters
Row 6: 421 characters
Row 7: 421 characters
Row 8: 421 characters
Row 9: 421 characters
Row 10: 421 characters


In [3]:
import pandas as pd

# Read the first four documented YRBS variables from the fixed-width file
yrbss_test = pd.read_fwf(
    yrbss_path,
    colspecs=[
        (16, 17),  # Q1 - Age
        (17, 18),  # Q2 - Sex
        (18, 19),  # Q3 - Grade
        (19, 20),  # Q4 - Hispanic/Latino
    ],
    names=["Q1_age", "Q2_sex", "Q3_grade", "Q4_hispanic"],
    dtype="string"
)

print(yrbss_test.shape)
yrbss_test.head(10)

(20098, 4)


,Q1_age,Q2_sex,Q3_grade,Q4_hispanic
0,3,1,1,<NA>
1,4,2,1,2
2,5,2,3,2
3,6,1,2,2
4,3,2,1,2
5,5,2,1,2
6,6,2,3,2
7,4,1,1,2
8,4,2,1,2
9,6,1,3,2


In [4]:
# Inspect character positions around the mental health section

with open(yrbss_path, "r", encoding="utf-8", errors="replace") as f:
    first_line = f.readline().rstrip("\n")

for start in range(0, len(first_line), 25):
    end = min(start + 25, len(first_line))
    print(f"{start + 1:>3}-{end:<3}: {first_line[start:end]}")

  1-25 : XX              311   C  
 26-50 :    1.65 81.65441111311122
 51-75 : 1221221221121112111111111
 76-100: 11112111 1121111111212123
101-125: 11112412216221131 2 11211
126-150: 3313121233222            
151-175:                          
176-200:          11  221222222222
201-225: 2212222222 22 22 2222 222
226-250: 1222 222222      21221111
251-275: 2221121221222 2 2212 2221
276-300: 221222221                
301-325:                          
326-350:                         2
351-375: 22222222222    2121222222
376-400: 122222222 12    0.8614103
401-421:  1629497.08    505180


## Select Analysis Variables

The full YRBSS dataset contains measures across multiple health and behavioral domains. For this analysis, variables will be selected based on their relevance to adolescent mental health, risk factors, protective factors, and demographic context.

In [5]:
selected_vars = {
    # Demographics
    "Q1": "age",
    "Q2": "sex",
    "Q3": "grade",
    "Q4": "hispanic_latino",

    # Mental health and suicidality
    "Q26": "persistent_sadness_hopelessness",
    "Q27": "considered_suicide",
    "Q28": "made_suicide_plan",
    "Q29": "attempted_suicide",
    "Q30": "suicide_attempt_injury"
}

selected_vars

{'Q1': 'age',
 'Q2': 'sex',
 'Q3': 'grade',
 'Q4': 'hispanic_latino',
 'Q26': 'persistent_sadness_hopelessness',
 'Q27': 'considered_suicide',
 'Q28': 'made_suicide_plan',
 'Q29': 'attempted_suicide',
 'Q30': 'suicide_attempt_injury'}

In [6]:
# Fixed-width positions for selected YRBSS variables
# Python uses 0-based indexing; CDC documentation uses 1-based positions.

colspecs = [
    (16, 17),  # Q1  - Age
    (17, 18),  # Q2  - Sex
    (18, 19),  # Q3  - Grade
    (19, 20),  # Q4  - Hispanic/Latino

    (56, 57),  # Q26 - Sad or hopeless
    (57, 58),  # Q27 - Considered suicide
    (58, 59),  # Q28 - Made suicide plan
    (59, 60),  # Q29 - Attempted suicide
    (60, 61),  # Q30 - Injurious suicide attempt
]

yrbss_df = pd.read_fwf(
    yrbss_path,
    colspecs=colspecs,
    names=list(selected_vars.values()),
    dtype="string"
)

print(yrbss_df.shape)
yrbss_df.head()

(20102, 9)


,age,sex,grade,hispanic_latino,persistent_sadness_hopelessness,considered_suicide,made_suicide_plan,attempted_suicide,suicide_attempt_injury
0,3,1,1,<NA>,1,2,2,1,1
1,4,2,1,2,2,2,2,1,1
2,5,2,3,2,1,2,2,1,1
3,6,1,2,2,1,2,2,1,1
4,3,2,1,2,1,2,1,1,1


In [7]:
# Count the raw records directly

with open(yrbss_path, "r", encoding="utf-8", errors="replace") as f:
    raw_lines = f.readlines()

print("Raw lines:", len(raw_lines))
print("Blank lines:", sum(1 for line in raw_lines if not line.strip()))
print("Unique record lengths:", sorted(set(len(line.rstrip("\n")) for line in raw_lines)))

Raw lines: 20103
Blank lines: 0
Unique record lengths: [421]


In [8]:
yrbss_df = pd.read_fwf(
    yrbss_path,
    colspecs=colspecs,
    names=list(selected_vars.values()),
    dtype="string",
    skip_blank_lines=False
)

print(yrbss_df.shape)

(20103, 9)


## Validate Selected Variables

Before expanding the dataset, review the selected variables for expected response codes, missing values, and overall completeness.

In [9]:
# Review value distributions for the selected variables

for column in yrbss_df.columns:
    print(f"\n{column}")
    print(yrbss_df[column].value_counts(dropna=False).sort_index())


age
age
1         44
2         33
3       2569
4       5526
5       5208
6       4458
7       2167
<NA>      98
Name: count, dtype: Int64

sex
sex
1        9884
2       10061
<NA>      158
Name: count, dtype: Int64

grade
grade
1       5680
2       5410
3       4811
4       3961
5         48
<NA>     193
Name: count, dtype: Int64

hispanic_latino
hispanic_latino
1        3997
2       15852
<NA>      254
Name: count, dtype: Int64

persistent_sadness_hopelessness
persistent_sadness_hopelessness
1        8108
2       11755
<NA>      240
Name: count, dtype: Int64

considered_suicide
considered_suicide
1        4214
2       15453
<NA>      436
Name: count, dtype: Int64

made_suicide_plan
made_suicide_plan
1        3212
2       15154
<NA>     1737
Name: count, dtype: Int64

attempted_suicide
attempted_suicide
1       17341
2        1109
3         630
4         100
5         156
<NA>      767
Name: count, dtype: Int64

suicide_attempt_injury
suicide_attempt_injury
1       13143
2         372

In [10]:
# YRBSS domains to retain for the pediatric mental health analysis

yrbss_domains = {
    "demographics": [
        "age",
        "sex",
        "grade",
        "race_ethnicity"
    ],

    "mental_health": [
        "poor_mental_health",
        "persistent_sadness_hopelessness",
        "considered_suicide",
        "made_suicide_plan",
        "attempted_suicide",
        "suicide_attempt_injury"
    ],

    "school_peer_context": [
        "bullied_at_school",
        "electronically_bullied",
        "missed_school_due_to_safety",
        "school_connectedness"
    ],

    "family_protective_factors": [
        "adult_met_basic_needs",
        "parental_monitoring"
    ],

    "behavior_lifestyle": [
        "sleep",
        "physical_activity",
        "sports_team",
        "social_media_use"
    ],

    "additional_context": [
        "experienced_racism_at_school",
        "unfair_discipline_at_school"
    ],

    "survey_design": [
        "weight",
        "stratum",
        "psu"
    ]
}

yrbss_domains

{'demographics': ['age', 'sex', 'grade', 'race_ethnicity'],
 'mental_health': ['poor_mental_health',
  'persistent_sadness_hopelessness',
  'considered_suicide',
  'made_suicide_plan',
  'attempted_suicide',
  'suicide_attempt_injury'],
 'school_peer_context': ['bullied_at_school',
  'electronically_bullied',
  'missed_school_due_to_safety',
  'school_connectedness'],
 'family_protective_factors': ['adult_met_basic_needs', 'parental_monitoring'],
 'behavior_lifestyle': ['sleep',
  'physical_activity',
  'sports_team',
  'social_media_use'],
 'additional_context': ['experienced_racism_at_school',
  'unfair_discipline_at_school'],
 'survey_design': ['weight', 'stratum', 'psu']}

## Build Curated YRBSS Dataset

Using the 2023 YRBSS codebook, import the selected mental health, school, family, behavioral, demographic, and survey design variables from their documented fixed-width positions.

In [11]:
# Selected 2023 YRBSS variables and documented ASCII locations
# CDC locations are 1-based; pd.read_fwf() uses 0-based indexing.

yrbss_columns = {
    # Demographics
    "age": (16, 17),                    # Q1
    "sex": (17, 18),                    # Q2
    "grade": (18, 19),                  # Q3
    "race_ethnicity": (413, 415),       # RACEETH

    # Mental health and suicidality
    "persistent_sadness_hopelessness": (56, 57),  # Q26
    "considered_suicide": (57, 58),               # Q27
    "made_suicide_plan": (58, 59),                 # Q28
    "attempted_suicide": (59, 60),                 # Q29
    "suicide_attempt_injury": (60, 61),            # Q30
    "poor_mental_health": (114, 115),              # Q84

    # School and peer context
    "missed_school_due_to_safety": (44, 45),       # Q14
    "experienced_racism_at_school": (53, 54),      # Q23
    "bullied_at_school": (54, 55),                 # Q24
    "electronically_bullied": (55, 56),            # Q25
    "school_connectedness": (133, 134),            # Q103
    "unfair_discipline_at_school": (135, 136),     # Q105

    # Family protective factors
    "adult_met_basic_needs": (129, 130),           # Q99
    "parental_monitoring": (134, 135),              # Q104

    # Behavior and lifestyle
    "physical_activity": (106, 107),                # Q76
    "sports_team": (108, 109),                      # Q78
    "social_media_use": (110, 111),                 # Q80
    "sleep": (115, 116),                            # Q85

    # Survey design
    "weight": (387, 397),                           # WEIGHT
    "stratum": (397, 400),                          # STRATUM
    "psu": (400, 406),                              # PSU
}

colspecs_curated = list(yrbss_columns.values())
column_names = list(yrbss_columns.keys())

curated_yrbss = pd.read_fwf(
    yrbss_path,
    colspecs=colspecs_curated,
    names=column_names,
    dtype="string",
    skip_blank_lines=False
)

print(curated_yrbss.shape)
curated_yrbss.head()

(20103, 25)


,age,sex,grade,race_ethnicity,persistent_sadness_hopelessness,considered_suicide,made_suicide_plan,attempted_suicide,suicide_attempt_injury,poor_mental_health,...,unfair_discipline_at_school,adult_met_basic_needs,parental_monitoring,physical_activity,sports_team,social_media_use,sleep,weight,stratum,psu
0,3,1,1,<NA>,1,2,2,1,1,1,...,2,1,3,1,2,6,3,0.8614,103,16294
1,4,2,1,5,2,2,2,1,1,3,...,2,5,5,5,2,4,5,0.8920,103,16294
2,5,2,3,5,1,2,2,1,1,2,...,2,5,4,8,1,8,1,0.5081,103,16294
3,6,1,2,5,1,2,2,1,1,3,...,2,5,5,3,2,8,4,1.1759,103,16294
4,3,2,1,5,1,2,1,1,1,3,...,2,5,5,8,1,6,3,0.8920,103,16294


## Clean and Recode Variables
Convert YRBSS coded responses into analysis-ready labels while preserving the original survey meaning and survey design fields.

In [12]:
for column in curated_yrbss.columns:
    print(f"\n{column}")
    print(
        curated_yrbss[column]
        .value_counts(dropna=False)
        .sort_index()
    )


age
age
1         44
2         33
3       2569
4       5526
5       5208
6       4458
7       2167
<NA>      98
Name: count, dtype: Int64

sex
sex
1        9884
2       10061
<NA>      158
Name: count, dtype: Int64

grade
grade
1       5680
2       5410
3       4811
4       3961
5         48
<NA>     193
Name: count, dtype: Int64

race_ethnicity
race_ethnicity
1       1334
2        995
3       1791
4        105
5       9700
6       1208
7       2786
8       1814
<NA>     370
Name: count, dtype: Int64

persistent_sadness_hopelessness
persistent_sadness_hopelessness
1        8108
2       11755
<NA>      240
Name: count, dtype: Int64

considered_suicide
considered_suicide
1        4214
2       15453
<NA>      436
Name: count, dtype: Int64

made_suicide_plan
made_suicide_plan
1        3212
2       15154
<NA>     1737
Name: count, dtype: Int64

attempted_suicide
attempted_suicide
1       17341
2        1109
3         630
4         100
5         156
<NA>      767
Name: count, dtype: Int64



In [13]:
# Keep respondents age 17 or younger
# YRBS age code 7 = 18 years old or older

yrbss_clean = curated_yrbss[
    curated_yrbss["age"] != "7"
].copy()

print("Before:", curated_yrbss.shape)
print("After:", yrbss_clean.shape)

Before: (20103, 25)
After: (17838, 25)


In [14]:
# Inspect respondents with missing age

missing_age = curated_yrbss[
    curated_yrbss["age"].isna()
].copy()

print("Missing-age records:", missing_age.shape)

# Count non-missing values across the selected variables
missing_age_summary = pd.DataFrame({
    "non_missing": missing_age.notna().sum(),
    "missing": missing_age.isna().sum(),
    "percent_complete": (missing_age.notna().mean() * 100).round(1)
})

missing_age_summary.sort_values("percent_complete", ascending=False)

Missing-age records: (98, 25)


,non_missing,missing,percent_complete
psu,98,0,100.0
weight,98,0,100.0
stratum,98,0,100.0
bullied_at_school,90,8,91.8
considered_suicide,89,9,90.8
sex,89,9,90.8
race_ethnicity,88,10,89.8
attempted_suicide,88,10,89.8
experienced_racism_at_school,87,11,88.8
electronically_bullied,87,11,88.8


In [15]:
# See how complete each individual missing-age record is

missing_age["non_missing_count"] = missing_age.notna().sum(axis=1)

missing_age["non_missing_count"].describe()

count    98.000000
mean     16.571429
std       3.955943
min       5.000000
25%      14.000000
50%      16.000000
75%      18.000000
max      24.000000
Name: non_missing_count, dtype: float64

In [16]:
# Restrict primary analysis to respondents with confirmed age 17 or younger
# Preserve missing-age records in curated_yrbss for potential secondary analysis

yrbss_clean = curated_yrbss[
    curated_yrbss["age"].notna() &
    (curated_yrbss["age"] != "7")
].copy()

print("Full curated dataset:", curated_yrbss.shape)
print("Primary age-restricted cohort:", yrbss_clean.shape)
print("Excluded for missing age:", curated_yrbss["age"].isna().sum())

Full curated dataset: (20103, 25)
Primary age-restricted cohort: (17838, 25)
Excluded for missing age: 98


In [17]:
yrbss_clean["age"].value_counts(dropna=False).sort_index()

age
1      44
2      33
3    2569
4    5526
5    5208
6    4458
Name: count, dtype: Int64

In [18]:
age_labels = {
    "1": "12 years old or younger",
    "2": "13 years old",
    "3": "14 years old",
    "4": "15 years old",
    "5": "16 years old",
    "6": "17 years old"
}

sex_labels = {
    "1": "Female",
    "2": "Male"
}

grade_labels = {
    "1": "9th grade",
    "2": "10th grade",
    "3": "11th grade",
    "4": "12th grade",
    "5": "Ungraded or other grade"
}

yrbss_clean["age"] = yrbss_clean["age"].map(age_labels)
yrbss_clean["sex"] = yrbss_clean["sex"].map(sex_labels)
yrbss_clean["grade"] = yrbss_clean["grade"].map(grade_labels)

yrbss_clean[
    ["age", "sex", "grade", "race_ethnicity"]
].head(10)

,age,sex,grade,race_ethnicity
0,14 years old,Female,9th grade,<NA>
1,15 years old,Male,9th grade,5
2,16 years old,Male,11th grade,5
3,17 years old,Female,10th grade,5
4,14 years old,Male,9th grade,5
5,16 years old,Male,9th grade,5
6,17 years old,Male,11th grade,5
7,15 years old,Female,9th grade,5
8,15 years old,Male,9th grade,8
9,17 years old,Female,11th grade,3


In [19]:
race_ethnicity_labels = {
    "1": "American Indian or Alaska Native",
    "2": "Asian",
    "3": "Black or African American",
    "4": "Native Hawaiian or Other Pacific Islander",
    "5": "White",
    "6": "Hispanic/Latino",
    "7": "Multiple race - Hispanic",
    "8": "Multiple race - Non-Hispanic"
}

yrbss_clean["race_ethnicity"] = (
    yrbss_clean["race_ethnicity"]
    .map(race_ethnicity_labels)
)

yrbss_clean["race_ethnicity"].value_counts(dropna=False)

race_ethnicity
White                                        8600
Multiple race - Hispanic                     2466
Multiple race - Non-Hispanic                 1626
Black or African American                    1595
American Indian or Alaska Native             1128
Hispanic/Latino                              1119
Asian                                         897
NaN                                           312
Native Hawaiian or Other Pacific Islander      95
Name: count, dtype: int64

In [20]:
yes_no_labels = {
    "1": "Yes",
    "2": "No"
}

for column in [
    "persistent_sadness_hopelessness",
    "considered_suicide",
    "made_suicide_plan"
]:
    yrbss_clean[column] = yrbss_clean[column].map(yes_no_labels)

In [21]:
yrbss_clean[
    [
        "persistent_sadness_hopelessness",
        "considered_suicide",
        "made_suicide_plan"
    ]
].head(10)

,persistent_sadness_hopelessness,considered_suicide,made_suicide_plan
0,Yes,No,No
1,No,No,No
2,Yes,No,No
3,Yes,No,No
4,Yes,No,Yes
5,No,No,No
6,Yes,No,No
7,No,No,No
8,Yes,Yes,No
9,Yes,NaN,Yes


In [22]:
# Add YRBSS ACE-related variables
# Positions use Python's 0-based indexing

ace_colspecs = [
    (49, 50),    # Q23 - Sexual abuse
    (50, 51),    # Q24 - Emotional abuse
    (51, 52),    # Q25 - Physical abuse
    (52, 53),    # Q26 - Witnessed intimate partner violence
    (130, 131),  # Q99 - Household substance use
    (131, 132),  # Q100 - Household poor mental health
    (132, 133),  # Q101 - Parent/guardian incarceration
]

ace_names = [
    "sexual_abuse",
    "emotional_abuse",
    "physical_abuse",
    "witnessed_intimate_partner_violence",
    "household_substance_use",
    "household_poor_mental_health",
    "parent_guardian_incarceration",
]

ace_df = pd.read_fwf(
    yrbss_path,
    colspecs=ace_colspecs,
    names=ace_names,
    dtype="string",
    skip_blank_lines=False
)

print(ace_df.shape)
ace_df.head()

(20103, 7)


,sexual_abuse,emotional_abuse,physical_abuse,witnessed_intimate_partner_violence,household_substance_use,household_poor_mental_health,parent_guardian_incarceration
0,2,1,2,2,2,1,2
1,2,1,2,2,2,2,2
2,2,1,2,2,2,2,1
3,1,1,2,2,2,2,2
4,2,1,2,1,2,2,2


In [23]:
# Add ACE variables to the age-restricted analysis cohort

for column in ace_names:
    yrbss_clean[column] = ace_df.loc[yrbss_clean.index, column]

print(yrbss_clean.shape)

(17838, 32)


In [24]:
# Add ACE variables to the age-restricted analysis cohort

for column in ace_names:
    yrbss_clean[column] = ace_df.loc[yrbss_clean.index, column]

print(yrbss_clean.shape)

(17838, 32)


In [25]:
# Add mental/emotional functioning measure: Q106
# CDC data location 137 = Python position (136, 137)

functioning_df = pd.read_fwf(
    yrbss_path,
    colspecs=[(136, 137)],
    names=["difficulty_concentrating_remembering_deciding"],
    dtype="string",
    skip_blank_lines=False
)

curated_yrbss["difficulty_concentrating_remembering_deciding"] = functioning_df[
    "difficulty_concentrating_remembering_deciding"
]

yrbss_clean["difficulty_concentrating_remembering_deciding"] = (
    functioning_df.loc[
        yrbss_clean.index,
        "difficulty_concentrating_remembering_deciding"
    ]
)

print(curated_yrbss.shape)
print(yrbss_clean.shape)

(20103, 26)
(17838, 33)


In [26]:
# Ensure ACE variables exist in both curated and cleaned datasets

for column in ace_names:
    curated_yrbss[column] = ace_df[column]
    yrbss_clean[column] = ace_df.loc[yrbss_clean.index, column]

# Ensure mental/emotional functioning exists in both

curated_yrbss["difficulty_concentrating_remembering_deciding"] = (
    functioning_df["difficulty_concentrating_remembering_deciding"]
)

yrbss_clean["difficulty_concentrating_remembering_deciding"] = (
    functioning_df.loc[
        yrbss_clean.index,
        "difficulty_concentrating_remembering_deciding"
    ]
)

print("Curated:", curated_yrbss.shape)
print("Clean:", yrbss_clean.shape)

Curated: (20103, 33)
Clean: (17838, 33)


In [27]:
# Confirm both datasets contain the same variables

print("Curated:", curated_yrbss.shape)
print("Clean:", yrbss_clean.shape)

print("\nColumns only in curated:")
print(set(curated_yrbss.columns) - set(yrbss_clean.columns))

print("\nColumns only in clean:")
print(set(yrbss_clean.columns) - set(curated_yrbss.columns))

Curated: (20103, 33)
Clean: (17838, 33)

Columns only in curated:
set()

Columns only in clean:
set()


In [28]:
new_vars = [
    "sexual_abuse",
    "emotional_abuse",
    "physical_abuse",
    "witnessed_intimate_partner_violence",
    "household_substance_use",
    "household_poor_mental_health",
    "parent_guardian_incarceration",
    "difficulty_concentrating_remembering_deciding"
]

for column in new_vars:
    print(f"\n{column}")
    print(
        yrbss_clean[column]
        .value_counts(dropna=False)
        .sort_index()
    )


sexual_abuse
sexual_abuse
1        1404
2       13763
<NA>     2671
Name: count, dtype: Int64

emotional_abuse
emotional_abuse
1       12213
2         689
3         561
4         137
5         241
<NA>     3997
Name: count, dtype: Int64

physical_abuse
physical_abuse
1        1994
2       10909
3         344
4         259
5          57
6         130
<NA>     4145
Name: count, dtype: Int64

witnessed_intimate_partner_violence
witnessed_intimate_partner_violence
1       7061
2       8979
3        438
4        305
5         93
6        226
<NA>     736
Name: count, dtype: Int64

household_substance_use
household_substance_use
1       3375
2       8297
<NA>    6166
Name: count, dtype: Int64

household_poor_mental_health
household_poor_mental_health
1       3775
2       7961
<NA>    6102
Name: count, dtype: Int64

parent_guardian_incarceration
parent_guardian_incarceration
1       2083
2       9676
<NA>    6079
Name: count, dtype: Int64

difficulty_concentrating_remembering_deciding
diffic

In [29]:
mental_health_context_vars = [
    "persistent_sadness_hopelessness",
    "poor_mental_health",
    "considered_suicide",
    "made_suicide_plan",
    "attempted_suicide",
    "suicide_attempt_injury",
    "difficulty_concentrating_remembering_deciding",
    "school_connectedness",
    "bullied_at_school",
    "electronically_bullied",
    "missed_school_due_to_safety",
    "experienced_racism_at_school",
    "unfair_discipline_at_school",
    "adult_met_basic_needs",
    "parental_monitoring",
    "sleep",
    "physical_activity",
    "sports_team",
    "social_media_use"
]

missing_mh_vars = [
    var for var in mental_health_context_vars
    if var not in yrbss_clean.columns
]

print("Expected mental-health/context variables:", len(mental_health_context_vars))
print("Missing:", missing_mh_vars)

Expected mental-health/context variables: 19
Missing: []


In [30]:
ace_vars = [
    "sexual_abuse",
    "emotional_abuse",
    "physical_abuse",
    "witnessed_intimate_partner_violence",
    "adult_met_basic_needs",
    "household_substance_use",
    "household_poor_mental_health",
    "parent_guardian_incarceration",
]

for column in ace_vars:
    print(f"\n{column}")
    print(
        yrbss_clean[column]
        .value_counts(dropna=False)
        .sort_index()
    )


sexual_abuse
sexual_abuse
1        1404
2       13763
<NA>     2671
Name: count, dtype: Int64

emotional_abuse
emotional_abuse
1       12213
2         689
3         561
4         137
5         241
<NA>     3997
Name: count, dtype: Int64

physical_abuse
physical_abuse
1        1994
2       10909
3         344
4         259
5          57
6         130
<NA>     4145
Name: count, dtype: Int64

witnessed_intimate_partner_violence
witnessed_intimate_partner_violence
1       7061
2       8979
3        438
4        305
5         93
6        226
<NA>     736
Name: count, dtype: Int64

adult_met_basic_needs
adult_met_basic_needs
1         895
2         378
3         622
4        1751
5       10335
<NA>     3857
Name: count, dtype: Int64

household_substance_use
household_substance_use
1       3375
2       8297
<NA>    6166
Name: count, dtype: Int64

household_poor_mental_health
household_poor_mental_health
1       3775
2       7961
<NA>    6102
Name: count, dtype: Int64

parent_guardian_incarce

In [31]:
# Replace ACE variables with corrected 2023 YRBS positions
# CDC locations are 1-based; Python positions are 0-based

correct_ace_colspecs = [
    (118, 119),  # Q88  - Sexual abuse
    (119, 120),  # Q89  - Emotional abuse
    (120, 121),  # Q90  - Physical abuse
    (121, 122),  # Q91  - Witnessed violence between adults in home
    (130, 131),  # Q100 - Household substance use
    (131, 132),  # Q101 - Household poor mental health
    (132, 133),  # Q102 - Parent/guardian incarceration
]

correct_ace_names = [
    "sexual_abuse",
    "emotional_abuse",
    "physical_abuse",
    "witnessed_intimate_partner_violence",
    "household_substance_use",
    "household_poor_mental_health",
    "parent_guardian_incarceration",
]

correct_ace_df = pd.read_fwf(
    yrbss_path,
    colspecs=correct_ace_colspecs,
    names=correct_ace_names,
    dtype="string",
    skip_blank_lines=False
)

for column in correct_ace_names:
    curated_yrbss[column] = correct_ace_df[column]
    yrbss_clean[column] = correct_ace_df.loc[yrbss_clean.index, column]

print("Curated:", curated_yrbss.shape)
print("Clean:", yrbss_clean.shape)

Curated: (20103, 33)
Clean: (17838, 33)


In [32]:
ace_vars = [
    "sexual_abuse",
    "emotional_abuse",
    "physical_abuse",
    "witnessed_intimate_partner_violence",
    "adult_met_basic_needs",
    "household_substance_use",
    "household_poor_mental_health",
    "parent_guardian_incarceration",
]

for column in ace_vars:
    print(f"\n{column}")
    print(
        yrbss_clean[column]
        .value_counts(dropna=False)
        .sort_index()
    )


sexual_abuse
sexual_abuse
1        1015
2       11155
<NA>     5668
Name: count, dtype: Int64

emotional_abuse
emotional_abuse
1       4728
2       3198
3       2760
4       1093
5        475
<NA>    5584
Name: count, dtype: Int64

physical_abuse
physical_abuse
1       8276
2       2465
3       1169
4        231
5        110
<NA>    5587
Name: count, dtype: Int64

witnessed_intimate_partner_violence
witnessed_intimate_partner_violence
1       9482
2       1513
3        813
4        213
5         93
<NA>    5724
Name: count, dtype: Int64

adult_met_basic_needs
adult_met_basic_needs
1         895
2         378
3         622
4        1751
5       10335
<NA>     3857
Name: count, dtype: Int64

household_substance_use
household_substance_use
1       3375
2       8297
<NA>    6166
Name: count, dtype: Int64

household_poor_mental_health
household_poor_mental_health
1       3775
2       7961
<NA>    6102
Name: count, dtype: Int64

parent_guardian_incarceration
parent_guardian_incarceration
1 

In [33]:
# Preserve full ACE response detail

frequency_labels = {
    "1": "Never",
    "2": "Rarely",
    "3": "Sometimes",
    "4": "Most of the time",
    "5": "Always"
}

yes_no_labels = {
    "1": "Yes",
    "2": "No"
}

# Binary ACE measures
for column in [
    "sexual_abuse",
    "household_substance_use",
    "household_poor_mental_health",
    "parent_guardian_incarceration"
]:
    yrbss_clean[column] = yrbss_clean[column].map(yes_no_labels)

# Frequency-based ACE measures
for column in [
    "emotional_abuse",
    "physical_abuse",
    "witnessed_intimate_partner_violence"
]:
    yrbss_clean[column] = yrbss_clean[column].map(frequency_labels)

In [34]:
basic_needs_labels = {
    "1": "Never",
    "2": "Rarely",
    "3": "Sometimes",
    "4": "Most of the time",
    "5": "Always"
}

yrbss_clean["adult_met_basic_needs"] = (
    yrbss_clean["adult_met_basic_needs"]
    .map(basic_needs_labels)
)

yrbss_clean["adult_met_basic_needs"].value_counts(dropna=False)

adult_met_basic_needs
Always              10335
NaN                  3857
Most of the time     1751
Never                 895
Sometimes             622
Rarely                378
Name: count, dtype: int64

In [35]:
# Recode remaining YRBSS variables while preserving full response detail

remaining_labels = {

    # Suicide
    "attempted_suicide": {
        "1": "0 times",
        "2": "1 time",
        "3": "2 or 3 times",
        "4": "4 or 5 times",
        "5": "6 or more times"
    },

    "suicide_attempt_injury": {
        "1": "Did not attempt suicide",
        "2": "Yes",
        "3": "No"
    },

    # Mental health
    "poor_mental_health": {
        "1": "Never",
        "2": "Rarely",
        "3": "Sometimes",
        "4": "Most of the time",
        "5": "Always"
    },

    "difficulty_concentrating_remembering_deciding": {
        "1": "Yes",
        "2": "No"
    },

    # School / peer context
    "missed_school_due_to_safety": {
        "1": "0 days",
        "2": "1 day",
        "3": "2 or 3 days",
        "4": "4 or 5 days",
        "5": "6 or more days"
    },

    "bullied_at_school": {
        "1": "Yes",
        "2": "No"
    },

    "electronically_bullied": {
        "1": "Yes",
        "2": "No"
    },

    "experienced_racism_at_school": {
        "1": "Never",
        "2": "Rarely",
        "3": "Sometimes",
        "4": "Most of the time",
        "5": "Always"
    },

    "school_connectedness": {
        "1": "Strongly agree",
        "2": "Agree",
        "3": "Not sure",
        "4": "Disagree",
        "5": "Strongly disagree"
    },

    "unfair_discipline_at_school": {
        "1": "Yes",
        "2": "No"
    },

    # Family / protective context
    "parental_monitoring": {
        "1": "Never",
        "2": "Rarely",
        "3": "Sometimes",
        "4": "Most of the time",
        "5": "Always"
    },

    # Lifestyle / protective factors
    "sleep": {
        "1": "4 or less hours",
        "2": "5 hours",
        "3": "6 hours",
        "4": "7 hours",
        "5": "8 hours",
        "6": "9 hours",
        "7": "10 or more hours"
    },

    "physical_activity": {
        "1": "0 days",
        "2": "1 day",
        "3": "2 days",
        "4": "3 days",
        "5": "4 days",
        "6": "5 days",
        "7": "6 days",
        "8": "7 days"
    },

    "sports_team": {
        "1": "0 teams",
        "2": "1 team",
        "3": "2 teams",
        "4": "3 or more teams"
    },

    "social_media_use": {
        "1": "I do not use social media",
        "2": "A few times a month",
        "3": "About once a week",
        "4": "A few times a week",
        "5": "About once a day",
        "6": "Several times a day",
        "7": "About once an hour",
        "8": "More than once an hour"
    }
}

# .replace() changes coded values while leaving any already-labeled values alone
for column, labels in remaining_labels.items():
    yrbss_clean[column] = yrbss_clean[column].replace(labels)

print("Recoded variables:", len(remaining_labels))
print("Dataset shape:", yrbss_clean.shape)

Recoded variables: 15
Dataset shape: (17838, 33)


In [36]:
# Check for any numeric survey codes remaining in recoded analysis variables

for column in remaining_labels:
    print(f"\n{column}")
    print(yrbss_clean[column].value_counts(dropna=False))


attempted_suicide
attempted_suicide
0 times            15371
1 time               982
<NA>                 692
2 or 3 times         566
6 or more times      134
4 or 5 times          93
Name: count, dtype: Int64

suicide_attempt_injury
suicide_attempt_injury
Did not attempt suicide    11509
<NA>                        5022
No                           981
Yes                          326
Name: count, dtype: Int64

poor_mental_health
poor_mental_health
Sometimes           4172
<NA>                3971
Most of the time    3006
Rarely              2931
Never               2479
Always              1279
Name: count, dtype: Int64

difficulty_concentrating_remembering_deciding
difficulty_concentrating_remembering_deciding
<NA>    8375
No      4875
Yes     4588
Name: count, dtype: Int64

missed_school_due_to_safety
missed_school_due_to_safety
0 days            14815
1 day               994
<NA>                992
2 or 3 days         592
6 or more days      293
4 or 5 days         152
Name: co

## Final Validation and Export

Validate the cleaned YRBSS analysis dataset, confirm the age-restricted cohort and retained survey design fields, review missingness, and save the final analysis-ready file.

In [37]:
# Final structure checks

print("Final cleaned shape:", yrbss_clean.shape)
print("Duplicate rows:", yrbss_clean.duplicated().sum())

print("\nMissing survey design fields:")
print(
    yrbss_clean[
        ["weight", "stratum", "psu"]
    ].isna().sum()
)

print("\nAge categories:")
print(yrbss_clean["age"].value_counts(dropna=False))

Final cleaned shape: (17838, 33)
Duplicate rows: 150

Missing survey design fields:
weight     0
stratum    0
psu        0
dtype: int64

Age categories:
age
15 years old               5526
16 years old               5208
17 years old               4458
14 years old               2569
12 years old or younger      44
13 years old                 33
Name: count, dtype: int64


In [38]:
# Check whether apparent duplicates are true duplicates in the raw YRBS file

with open(yrbss_path, "r", encoding="utf-8", errors="replace") as f:
    raw_records = [line.rstrip("\n") for line in f]

# Use the same row indices as the age-restricted analysis cohort
clean_raw_records = pd.Series(
    [raw_records[i] for i in yrbss_clean.index],
    index=yrbss_clean.index
)

print("Duplicate rows in cleaned 33-column dataset:",
      yrbss_clean.duplicated().sum())

print("Exact duplicate full raw records:",
      clean_raw_records.duplicated().sum())

Duplicate rows in cleaned 33-column dataset: 150
Exact duplicate full raw records: 0


In [39]:
# Final missingness summary

missingness_summary = pd.DataFrame({
    "missing_count": yrbss_clean.isna().sum(),
    "missing_percent": (yrbss_clean.isna().mean() * 100).round(1)
}).sort_values("missing_percent", ascending=False)

missingness_summary

,missing_count,missing_percent
unfair_discipline_at_school,8688,48.7
parental_monitoring,8454,47.4
difficulty_concentrating_remembering_deciding,8375,47.0
school_connectedness,8158,45.7
sports_team,6578,36.9
household_substance_use,6166,34.6
household_poor_mental_health,6102,34.2
parent_guardian_incarceration,6079,34.1
witnessed_intimate_partner_violence,5724,32.1
sexual_abuse,5668,31.8


In [40]:
from pathlib import Path

output_dir = Path("../data/cleaned")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "yrbss_2023_cleaned.csv"

yrbss_clean.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Final shape:", yrbss_clean.shape)

Saved: ..\data\cleaned\yrbss_2023_cleaned.csv
Final shape: (17838, 33)
